> **Important parameters of `StateDependentWalker`**
>
> The walker can be initialized either from a `pandas.DataFrame` or from a `movingpandas.TrajectoryCollection`.
>
> **Main constructor parameters**
>
> - **`data`**
>   Input trajectory data. This can be either raw tabular data or an existing `TrajectoryCollection`.
>
> - **`animal_type`**
>   Specifies the movement domain of the animal, e.g. terrestrial, aerial, or marine.
>   This affects how movement and environmental constraints are interpreted.
>
> - **`resolution`**
>   Spatial grid resolution used for discretizing the environment and generating walks.
>   Conceptually, this controls how fine-grained the simulated movement space is.
>
> - **`out_directory`**
>   Output directory for generated kernels, intermediate files, and simulated walks.
>
> - **`n_hmm_states` / `hmm_states`**
>   Number of hidden states used in the HMM.
>   These states represent different movement modes and are later used to build state-dependent kernels.
>
> **Parameters for walk generation**
>
> - **`dt_tolerance`**
>   Tolerance for irregular sampling intervals.
>   It is used to decide whether two consecutive records still belong to the same trajectory segment.
>   For regular data, a value such as `2` may mean that time deviations up to about twice the median `Δt` are still accepted.
>
> - **`rnge` / `kernel_range`**
>   Spatial range of the generated kernels.
>   Larger values allow movement kernels to cover a wider neighborhood.
>
> - **`movement_policy`**
>   Determines how the temporal and spatial discretization of the simulated walk is chosen.
>
> - **`max_cell_size`**
>   Maximum number of grid cells that may be traversed in one simulated movement step.
>
> - **`water_mode`**
>   Controls how water is handled for terrestrial animals:
>   - `FORBID`: water cannot be crossed
>   - `AVOID`: water is penalized/avoided, but crossing may still be possible
>   - relaxed handling: water is treated like normal terrain
>
> - **`is_brownian`**
>   Controls whether Brownian-type kernels are used when generating the state-dependent movement model.

In [ ]:
import pickle
import movingpandas as mpd
import numpy as np
from random_walk_package.bindings.plotter import plot_animal_segments_overview, plot_walk_terrain
from random_walk_package.bindings import landcover_to_discrete_ptr
from random_walk_package.utils.geo_transformations import padded_utm_bbox, grid_shape_from_bbox, utm_to_grid
from random_walk_package.utils.trajectory_segmentation import trajectory_segments, merge_singletons, make_overlapping, \
    bbox_of_segment


from random_walk_package import Animal, StateDependentWalker, TimeStepPolicy, WaterMode, FixedStepsPolicy

input_file = "boars_in.pickle"
out_dir = "."

with open(input_file, "rb") as f:
    data:mpd.TrajectoryCollection = pickle.load(f)

animal_type = Animal.TERRESTRIAL
water_mode = WaterMode.AVOID
resolution = 200
hmm_states = 3
kernel_range = 300
dt_tolerance = 2
is_brownian = True
max_cell_size = 2
mvm_pol = FixedStepsPolicy(10)

### HMM-Parameters
> The data structure passed to the HMM contains:
>
> - spatial point coordinates (UTM)
> - trajectory ID
> - time information
> - speed
> - direction
> - angular difference
> - distance
> - angular diffusivity
> - terrain
>
> The HMM uses this processed trajectory table to infer `num_states` hidden movement states, which are then used to compute state-dependent kernels.

In [ ]:
walker = StateDependentWalker(data=data,
                              animal_type=Animal.TERRESTRIAL,
                              resolution=600,
                              out_directory=out_dir,
                              n_hmm_states=hmm_states)
py_kernels = walker.get_kernels(dt_tolerance, kernel_range, out_dir, is_brownian=is_brownian)
NUM_STATES = len(py_kernels)

> **Segmentation logic**
>
> Let the trajectory be a sequence of points
>
> $$
 p_i = (x_i, y_i), \quad i = 0,1,\dots,n-1
> $$
>
> and define the distance threshold
>
> $$
 r_{\max} = \frac{\texttt{max\_cell\_size} \cdot \texttt{resolution}}{4}
 $$
>
> A segment starts at index \(s\) with reference point
>
> $$
p_s = (x_s, y_s).
$$
>
> For each following point \(p_i\), its Euclidean distance to the current reference point is computed as
>
> $$
d(p_s, p_i) = \sqrt{(x_i - x_s)^2 + (y_i - y_s)^2}.
 $$
>
> As long as
>
> $$
d(p_s, p_i) < r_{\max},
 $$
>
> the point remains in the current segment.
>
> A new segment is started as soon as
>
> $$
d(p_s, p_i) \ge r_{\max}.
 $$
>
> In that case, the current segment is closed as
>
> $$
(s, i-1),
$$
>
> and a new segment begins at index \(i\). The new reference point is then set to \(p_i\).
>
> After all points have been processed, the final segment
>
> $$
(s, n-1)
$$
>
> is appended.

In [ ]:
steps_dict = walker.get_steps()

per_animal_gdfs = []
aid = 0
for animal_id, trajectory in steps_dict.items():
    print(f"ANIMAL: {animal_id}\n")
    aid += 1
    steps = trajectory.df
    steps_geo = np.array([[p.x, p.y] for p in steps.geometry], dtype=float)
    # segments as index-intervals with overlaps, e.g. [(0, 3), (3, 5), (5, 9)].
    segments = trajectory_segments(steps, max_cell_size, resolution)
    segments = merge_singletons(segments)
    segments = make_overlapping(segments)

    min_lon, min_lat, max_lon, max_lat = bbox_of_segment(steps, [0, len(steps) - 1])
    utm_bbox, zone, hemi, epsg_code, fwd, inv = padded_utm_bbox(
            min_lon, min_lat, max_lon, max_lat,
            padding=0.2,
            max_cell_size=max_cell_size
        )
    Nx, Ny = grid_shape_from_bbox(utm_bbox, resolution)
    terrain = landcover_to_discrete_ptr(file_path=walker.animal_proc.terrain_TIFFs[str(animal_id)],
                                                res_x=Nx, res_y=Ny,
                                                min_lon=min_lon, min_lat=min_lat,
                                                max_lon=max_lon, max_lat=max_lat)
    plot_walk_terrain(terrain, steps_geo, Nx, Ny)
    plot_animal_segments_overview(
        steps,
        segments,
        animal_id=animal_id,
        show_segment_lines=True,
        show_step_points=False,
        use_padded_bbox=False,
        max_cell_size=max_cell_size,
        resolution=resolution,
    )

### For Birds: Subsampling entire Kernels

In [ ]:
from skimage.transform import resize
import matplotlib.pyplot as plt

def plot(kernel, title):
    plt.figure(figsize=(6, 6))
    plt.imshow(kernel, origin="lower")
    plt.colorbar(label="probability")
    plt.title(title)
    plt.tight_layout()
    plt.show()
    plt.close()

S = 10
state = 1
target = 2 * S + 1

plot(py_kernels[state], f"Original Kernel range={kernel_range}")

grid_kernel = resize(
    py_kernels[state],
    (target, target),
    order=1,
    mode="reflect",
    anti_aliasing=True,
    preserve_range=True
)
grid_kernel = np.maximum(grid_kernel, 0)
grid_kernel /= grid_kernel.sum()
plot(grid_kernel, f"Subsampled Kernel S={S}")

### For Terrestrial: Clip and subsample

In [ ]:
from random_walk_package.utils.walker_utils import resample_kernel_to_grid
from random_walk_package.bindings.data_structures.kernels import normalize_kernel, clip_kernel

S = 100

plot(py_kernels[state], "Original Kernel")
kernel_radius = int(S * max_cell_size)
kernel_radius = min(kernel_range, kernel_radius)

clipped_kernel = normalize_kernel(clip_kernel(py_kernels[state], kernel_radius))
grid_kernel = resample_kernel_to_grid(clipped_kernel, max_cell_size, S)

plot(grid_kernel, f"Clipped [-{kernel_radius}, {kernel_radius}]")